## Imports

In [1]:
import json

import dateutil
import pandas as pd
from pandas.testing import assert_index_equal

from analysis_helpers.constants import PROCESSED_DIR, RAW_DIR

## Functions

In [2]:
def get_start_time(datastring):
    """
    Parses the timestamp for the first page of the pre-experiment 
    questionnaire from the psiTurk datastring. Used as a more accurate 
    replacement for "beginhit" field ("beginhit" is load time of 
    "welcome" screen, which was loaded before participant arrived).
    
    Parameters
    ----------
    datastring : 'dict'
        JSON object containing data saved by psiTurk experiment data
        
    Returns
    -------
    'int'
        Load time as a POSIX timestamp in milliseconds
    """
    return datastring['data'][0]['dateTime']


def parse_posix_ms(datetime_str):
    """
    Converts datetime strings from Google Form data to POSIX timestamps. 
    Google Forms converts all dates to current local time zone when 
    downloading, so participants collected during EDT show EST 
    equivalents. Pandas's default 'date_parser' doesn't handle this 
    behavior correctly and returns incorrect 'pandas.Timestamp's
    
    Parameters
    ----------
    datetime_str : 'str'
        tz-aware datetime string (format: YYYY/MM/DD H:MM:SS AM/PM EST)
        
    Returns
    -------
    'int'
        POSIX timestamp in milliseconds
    """
    dt = dateutil.parser.parse(datetime_str, tzinfos={'EST': -18000})
    return dt.timestamp() * 1000


def load_testroom_data(room):
    # load in psiturk data
    rmdf = pd.read_json(RAW_DIR.joinpath('db', 'exported', f'room{room}.json'))
    # drop debugging runs
    rmdf = rmdf[rmdf.status != 1].reset_index(drop=True)
    # drop final test run
    rmdf = rmdf.loc[1:]
    # keep only relevant columns
    rmdf = rmdf.filter(('uniqueid','datastring'))
    # load psiTurk data JSON string
    rmdf['datastring'] = rmdf['datastring'].apply(json.loads)
    # add back in "beginhit" column with more accurate timestamp
    rmdf['beginhit'] = rmdf['datastring'].apply(get_start_time)
    # make sure new "beginhit" value didn't change relative order
    assert_index_equal(rmdf.index, rmdf.sort_values('beginhit').index)
    # record test room number
    rmdf['testroom'] = room
    return rmdf


def load_google_form_data(filename):
    """
    Loads questionnaire data from Google Forms, converts timestamps, 
    dropts test runs (where Subject ID is NaN).
    """
    form_data_path = RAW_DIR.joinpath('google-form-data', f'{filename}.csv')
    gform_df = pd.read_csv(form_data_path)
    gform_df['Timestamp'] = gform_df['Timestamp'].apply(parse_posix_ms)
    return gform_df.dropna(subset=['Subject ID']).reset_index(drop=True)


def merge_with_duplicates(left, right, on=None, **kwargs):
    """
    Performs a 1:1 merge between two 'pandas.DataFrame's on columns with 
    non-unique keys. Row pairs from 'left' and 'right' containing 
    non-unique keys are matched by order (i.e., cumulative count of 
    key/key set occurrences) and non-unique row order is preserved in 
    the merged result. Thus for the result to be a proper 1:1 merge, 
    'left' and 'right' should contain the same number of duplicates of 
    each non-unique key per column being merged on.
    
    Parameters
    ----------
    left, right : 'pandas.DataFrame'
        DataFrames to merge
    on : label or iterable of labels
        Column name(s) to join on
    **kwargs
        Additional keyword arguments passed to `pandas.DataFrame.merge`
    
    Returns
    -------
    `pandas.DataFrame`
        Merged DataFrame object
        
    """
    _left, _right = left.copy(), right.copy()
    _left['_'] = _left.groupby(on).cumcount()
    _right['_'] = _right.groupby(on).cumcount()
    if isinstance(on, str):
        on = [on] + ['_']
    else:
        on = list(on) + ['_']
    return _left.merge(_right, on=on, **kwargs).drop(columns='_')

## Participants to drop from analyses

In [3]:
# ================== data collection stopped mid-task ==================
# Task & post-questionnaire data not recorded. Drop pre-questionnaire 
# data before merging
errors_session1 = [
    'MD-1011318-A-05',   # system backup during experiment caused crash
    'MD-020119-A-01',    # participant accidentally pressed key to exit
    'MD-020119-B-01'     # participant felt ill and chose to drop out
]

# (drop ses. 2 pre-questionnaire before merging, ses. 1 data after)
errors_session2 = [
    'MD-101218-B-04'     # Docker daemon crashed during session 2 task
]

# ================= data dropped after task completion =================
# Pre-questionnaire, task, & post-questionnaire data recorded. Drop all 
# data ater merging.
dropids = errors_session2 + [
    'MD-020119-B-03',    # participant did not return for session 2
    'MD-102218-B-06',    # participant did not return for session 2
    'MD-101318-A-01',    # participant did not return for session 2
    'MD-013119-A-01',    # participant did not return for session 2
    'MD-102218-B-04',    # participant did not return for session 2
    'MD-102318-A-01',    # participant did not return for session 2
    'MD-022019-B-01',    # participant did not return for session 2
    'MD-020719-B-01',    # reported speakers cutting out during task
    'MD-102218-A-05',    # mic stopped recording stopped during recall
    'MD-102218-A-04',    # self-reported task difficulty due to migraine    
]

## Load psiTurk data

In [4]:
# load psiTurk data for each testroom
rm1df = load_testroom_data(room=1)
rm2df = load_testroom_data(room=2)

# concatenate test room dataframes, order by start time
expdf = pd.concat((rm1df, rm2df)).sort_values('beginhit').reset_index(drop=True)
expdf.head()

,uniqueid,datastring,beginhit,testroom
0,debugIEH2T:debugDLVLJ,"{'condition': 0, 'counterbalance': 0, 'assignm...",1539368162836,1
1,debugBUnNA:debugLtZcs,"{'condition': 0, 'counterbalance': 0, 'assignm...",1539371956776,1
2,debugYQfMB:debugxg7il,"{'condition': 0, 'counterbalance': 0, 'assignm...",1539372566510,2
3,debugd1YD1:debug4FrAg,"{'condition': 0, 'counterbalance': 0, 'assignm...",1539375821845,1
4,debug92cgv:debugvdAIT,"{'condition': 0, 'counterbalance': 0, 'assignm...",1539376317256,2


## Load pre- & post-experiment questionnaire responses

In [5]:
# load, convert timestamps, exclude test runs
preqdf = load_google_form_data('pre-experiment-questionnaire')
postqdf = load_google_form_data('post-experiment-questionnaire')

# assign timestamp columns different names so both remain after merging
preqdf = preqdf.rename(columns={'Timestamp':'preqtime'})
postqdf = postqdf.rename(columns={'Timestamp':'postqtime'})

# drop participants with only session 1 pre-questionnaire data recorded
preqdf = preqdf[~preqdf['Subject ID'].isin(errors_session1)]

# drop session 2 pre-questionnaire if no session 2 data recorded
for subid in errors_session2:
    ses2_entry_row = preqdf.loc[preqdf['Subject ID'] == subid].index[-1]
    preqdf = preqdf.drop(ses2_entry_row)
    
preqdf = preqdf.reset_index(drop=True)

## merge task & questionnaire data

In [6]:
# all three datasets should have the same number of entries now
assert expdf.shape[0] == preqdf.shape[0] == postqdf.shape[0]

expdf = pd.concat([expdf, preqdf], axis=1)
expdf = merge_with_duplicates(expdf, postqdf, on='Subject ID')
expdf.head()

,uniqueid,datastring,beginhit,testroom,preqtime,Subject ID,"Outside of this study, have you ever watched an episode of either of the TV shows ""Atlanta"" or ""Arrested Development?""",Is English your first language?,Do you have any hearing or speech impairments?,Do you have normal color vision?,...,What is/was your major?,How many hours of sleep did you get last night?,How many cups of coffee have you had today?,How alert are you feeling?,postqtime,How engaging did you find the episode?,How easy/difficult was it to follow the episode?,How well do you feel you recalled the events of the episode?,How well do you feel you learned the characters' names over the course of the episode?,How tired do you feel?
0,debugIEH2T:debugDLVLJ,"{'condition': 0, 'counterbalance': 0, 'assignm...",1539368162836,1,1.539368e+12,MD-101218-A-01,I've never watched either one,Yes,No,Yes,...,undeclared,7.0,1.0,A little alert,1.539372e+12,Very engaging,Somewhat easy,Very well,Very well,A little tired
1,debugBUnNA:debugLtZcs,"{'condition': 0, 'counterbalance': 0, 'assignm...",1539371956776,1,1.539372e+12,MD-101218-B-01,I've never watched either one,Yes,No,Yes,...,undeclared,5.0,0.0,A little sluggish,1.539375e+12,A little engaging,Somewhat easy,Very well,Somewhat well,A little tired
2,debugYQfMB:debugxg7il,"{'condition': 0, 'counterbalance': 0, 'assignm...",1539372566510,2,1.539373e+12,MD-101218-A-02,I've never watched either one,Yes,No,Yes,...,undeclared,7.0,2.0,A little alert,1.539376e+12,Very engaging,Somewhat easy,Somewhat well,Somewhat well,A little tired
3,debugd1YD1:debug4FrAg,"{'condition': 0, 'counterbalance': 0, 'assignm...",1539375821845,1,1.539376e+12,MD-101218-B-02,I've never watched either one,Yes,No,Yes,...,neuroscience,9.0,0.0,A little alert,1.539379e+12,Very engaging,Very easy,Very well,Somewhat well,A little alert
4,debug92cgv:debugvdAIT,"{'condition': 0, 'counterbalance': 0, 'assignm...",1539376317256,2,1.539376e+12,MD-101218-A-03,I've never watched either one,Yes,No,Yes,...,Sociology,7.0,0.0,A little alert,1.539379e+12,Very engaging,Somewhat easy,Very well,Somewhat well,Very alert


## fix a few typos & drop excluded participants

In [7]:
# first, correct some typos in the Google Forms...
typos = {
    'MD--22819-B-01' : 'MD-022819-B-01',
    'MD-101218-A-06' : 'MD-102218-A-06',
    'MD-102118-B-06' : 'MD-102218-B-06',
    'MD-011319-A-02' : 'MD-013119-A-02'
}
expdf = expdf.replace(typos)

# ... and fix a repeated participant ID
expdf.loc[expdf.index[expdf['Subject ID'] == 'MD-101218-A-03'][2], 'Subject ID'] = 'MD-102018-A-03'

# drop excluded participants
expdf = expdf[~expdf['Subject ID'].isin(dropids)].reset_index(drop=True)

# all participant IDs should now appear in dataset exactly twice
assert (expdf['Subject ID'].value_counts() == 2).all()

## format & save across-session ID mappings for analyses

In [8]:
subid_mapping = {sid : tuple(expdf.loc[expdf['Subject ID'] == sid, 'uniqueid'].values) 
                 for sid in expdf['Subject ID'].unique()}
subid_mapping = pd.DataFrame.from_dict(subid_mapping, 
                                       orient='index', 
                                       columns=['session 1', 'session 2'])
subid_mapping.head()

,session 1,session 2
MD-101218-A-01,debugIEH2T:debugDLVLJ,debug2Ea7T:debugosNZ7
MD-101218-B-01,debugBUnNA:debugLtZcs,debugQEynG:debugpwxCU
MD-101218-A-02,debugYQfMB:debugxg7il,debugyBEnU:debugSXeyx
MD-101218-B-02,debugd1YD1:debug4FrAg,debugbu5Bq:debugl91xD
MD-101218-A-03,debug92cgv:debugvdAIT,debugIFCgX:debugt0bgV


In [9]:
# expdf.to_pickle(PROCESSED_DIR.joinpath('etc', 'expdf.p'))
# subid_mapping.to_pickle(PROCESSED_DIR.joinpath('etc', 'subid_mapping.p'))